# A/B testing and statistical power

An A/B test compares two variants on a binary outcome such as conversion. Running it well means
deciding the sample size in advance from a power calculation, not stopping the moment the result looks
significant. This notebook covers the two-proportion test, power and sample-size planning, the damage
that repeated peeking does to the false-positive rate, and the Bayesian alternative that reports the
probability one variant is better along with the expected cost of being wrong.


## The mathematics of peeking

A fixed-sample test controls the type-I error at $\alpha$ only if the decision is made once, at the
planned sample size. Repeatedly testing as data accrue and stopping at the first significant result
inflates the error.

Theorem (optional stopping inflates type-I error). Under the null, the running z-statistic behaves like a
random walk. By the law of the iterated logarithm it almost surely exceeds any fixed boundary infinitely
often, so a tester who keeps peeking will, with probability approaching one as the number of looks grows,
eventually cross the $\alpha$ threshold even though the null is true. The realized false-positive rate
tends to one, not $\alpha$.

Sketch. The partial sums $S_n$ of mean-zero increments satisfy $\limsup_n S_n/\sqrt{2n\log\log n}=1$ a.s.,
so $S_n$ recurrently reaches order $\sqrt{n\log\log n}$, which dwarfs the fixed $\sqrt n$ scale of a
constant z-threshold. Hence the boundary is crossed infinitely often under continuous monitoring.
$\quad\blacksquare$

The applied section measures this inflation directly; group-sequential boundaries (O'Brien-Fleming) or
mixture sequential probability ratio tests spend the error budget across looks to restore control.

Counterexample / contrast (Bayesian decisions). A Bayesian expected-loss rule does not have a frequentist
type-I rate by construction, but it is not immune to optional stopping either: thresholds on posterior
probability must still be calibrated by simulation if a frequentist guarantee is desired, the same lesson
as in the adaptive-trials notebook.

## 1. The two-proportion test

Variant B converts at a higher rate than A. The z-test for two proportions asks whether the observed
difference is larger than sampling noise would typically produce under the null of equal rates.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.RandomState(0)
nA = nB = 2000; pA, pB = 0.10, 0.12
a = rng.binomial(nA, pA); b = rng.binomial(nB, pB)
phatA, phatB = a / nA, b / nB
pool = (a + b) / (nA + nB)
se = np.sqrt(pool * (1 - pool) * (1 / nA + 1 / nB))
z = (phatB - phatA) / se; pval = 2 * stats.norm.sf(abs(z))
print('A: %d/%d = %.3f   B: %d/%d = %.3f' % (a, nA, phatA, b, nB, phatB))
print('z = %.2f, two-sided p-value = %.3f' % (z, pval))

A: 204/2000 = 0.102   B: 251/2000 = 0.126
z = 2.34, two-sided p-value = 0.019


## 2. Power and sample size

Power is the probability of detecting a real effect of a given size. Before running anything you fix
the effect you care about, the significance level, and the desired power, and solve for the sample size.
Under-powered tests waste traffic and mostly produce inconclusive or exaggerated results.


In [2]:
def sample_size(p0, p1, alpha=0.05, power=0.8):
    za, zb = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    pbar = (p0 + p1) / 2
    return ((za * np.sqrt(2 * pbar * (1 - pbar)) + zb * np.sqrt(p0 * (1 - p0) + p1 * (1 - p1))) ** 2
            / (p1 - p0) ** 2)
for effect in [0.02, 0.01, 0.005]:
    print('to detect %.1f%% -> %.1f%% you need ~%d users per arm (80%% power, alpha 0.05)' %
          (100 * 0.10, 100 * (0.10 + effect), int(np.ceil(sample_size(0.10, 0.10 + effect)))))
print('halving the effect you want to catch roughly quadruples the required sample size.')

to detect 10.0% -> 12.0% you need ~3841 users per arm (80% power, alpha 0.05)
to detect 10.0% -> 11.0% you need ~14751 users per arm (80% power, alpha 0.05)
to detect 10.0% -> 10.5% you need ~57763 users per arm (80% power, alpha 0.05)
halving the effect you want to catch roughly quadruples the required sample size.


## 3. The peeking problem

Checking significance repeatedly and stopping at the first p < 0.05 inflates the false-positive rate
far above 5%, because you give yourself many chances to cross the line by chance. We simulate A/A tests
(no true difference) with continuous peeking and measure the inflated error.


In [3]:
def peeking_fp(n_max=4000, checks=20, trials=2000):
    fp = 0
    pts = np.linspace(200, n_max, checks).astype(int)
    for _ in range(trials):
        xa = rng.random(n_max) < 0.10; xb = rng.random(n_max) < 0.10   # identical rates (null true)
        flagged = False
        for m in pts:
            pa, pb = xa[:m].mean(), xb[:m].mean(); pl = (pa + pb) / 2
            s = np.sqrt(pl * (1 - pl) * 2 / m)
            if s > 0 and 2 * stats.norm.sf(abs((pb - pa) / s)) < 0.05:
                flagged = True; break
        fp += flagged
    return fp / trials
print('false-positive rate with a single look at the end = ~0.05 (by construction)')
print('false-positive rate when peeking 20 times          = %.2f  (badly inflated)' % peeking_fp())

false-positive rate with a single look at the end = ~0.05 (by construction)


false-positive rate when peeking 20 times          = 0.23  (badly inflated)


## 4. Bayesian A/B testing

A Bayesian analysis puts a Beta posterior on each rate and answers the question decision-makers
actually ask: what is the probability B beats A, and if we ship B and we are wrong, how much do we lose
on average (the expected loss)?


In [4]:
postA = rng.beta(1 + a, 1 + nA - a, 200000)
postB = rng.beta(1 + b, 1 + nB - b, 200000)
p_b_better = (postB > postA).mean()
expected_loss_ship_B = np.mean(np.maximum(postA - postB, 0))   # loss if A was actually better
print('P(B > A | data) = %.3f' % p_b_better)
print('expected loss of shipping B = %.2e (conversion rate forgone if A was secretly better)' % expected_loss_ship_B)
print('decision rule: ship B when P(B>A) is high and the expected loss is below a tolerance.')

P(B > A | data) = 0.991
expected loss of shipping B = 3.16e-05 (conversion rate forgone if A was secretly better)
decision rule: ship B when P(B>A) is high and the expected loss is below a tolerance.


## References

- Wald, A. (1945). Sequential tests of statistical hypotheses. Annals of Mathematical Statistics.
- Kohavi, R., Tang, D. & Xu, Y. (2020). Trustworthy Online Controlled Experiments. Cambridge University Press.
- Johari, R., Koomen, P., Pekelis, L. & Walsh, D. (2017). Peeking at A/B tests: why it matters and what to do about it. KDD. https://doi.org/10.1145/3097983.3097992
- Stucchio, C. (2015). Bayesian A/B testing at VWO (expected-loss decision rule).


## Exercises

1. Implement a sequential test that controls the false-positive rate under peeking, either an alpha-spending boundary (O'Brien-Fleming) or the mixture sequential probability ratio test, and confirm the error returns to 5%.
2. Plot a full power curve: for a fixed sample size, sweep the true effect size and show the probability of detection rising from alpha to near 1.
3. Extend the Bayesian analysis to a decision with an explicit loss for each wrong call and find the stopping rule that minimizes expected loss subject to a traffic budget.
4. Simulate the winner's curse: among tests that just barely reach significance, show the estimated effect overstates the truth, and discuss shrinkage corrections.
